# Chapter 2 — Hallucination Is Not One Thing

**Book alignment:** Hallucination From First Principles, Chapter 2

**Question this notebook isolates:** Does a single response-level label hide failures that per-claim typed records separate?

Synthetic fixtures in this notebook demonstrate the mechanism type only and do not reproduce the book's 10k-row empirical run.


In [ ]:
import numpy as np

rng = np.random.default_rng(1)


## 1. One response label hides two different failures

Chapter 2 (sec. 3) decomposes a response into atomic claims because a response-level FAIL bit cannot say which part failed. We build two synthetic two-claim responses that share one response-level label but carry different typed per-claim records, including a relation reversal that preserves every entity.


In [ ]:
# Typed claim records: (reference, failure_relation) per Chapter 2 axes.
resp_X = [
    {"claim": "Marie Curie won the Nobel Prize in Physics in 1903.", "reference": "world", "verdict": "PASS", "failure": None},
    {"claim": "Dr. Elena Voss received the 2017 Northbridge Medal.", "reference": "world", "verdict": "FAIL", "failure": "invented_entity"},
]
resp_Y = [
    {"claim": "Company A acquired Company B from Company C in 2024.", "reference": "world", "verdict": "PASS", "failure": None},
    {"claim": "Company B acquired Company A from Company C in 2024.", "reference": "world", "verdict": "FAIL", "failure": "relation_reversal"},
]


def response_label(records):
    return "PASS" if all(r["verdict"] == "PASS" for r in records) else "FAIL"


lab_X, lab_Y = response_label(resp_X), response_label(resp_Y)
print("response X label:", lab_X, "|", [r["failure"] for r in resp_X])
print("response Y label:", lab_Y, "|", [r["failure"] for r in resp_Y])

# The reversal preserves entities exactly: same token multiset, different relation.
ev = "Company A acquired Company B from Company C in 2024"
cl = "Company B acquired Company A from Company C in 2024"
print("same entity set:", sorted(ev.split()) == sorted(cl.split()))
print("same relation:  ", ev == cl)


In [ ]:
assert lab_X == "FAIL" and lab_Y == "FAIL"  # response-level labels coincide
fails_X = [r["failure"] for r in resp_X if r["verdict"] == "FAIL"]
fails_Y = [r["failure"] for r in resp_Y if r["verdict"] == "FAIL"]
assert fails_X == ["invented_entity"]
assert fails_Y == ["relation_reversal"]
assert fails_X != fails_Y  # typed records separate what the scalar hides
assert sorted(ev.split()) == sorted(cl.split()) and ev != cl
print("response-level collision with distinct typed failures confirmed")


## 2. The taxonomy predicts the detector (and its blind spot)

Chapter 2 (sec. 12) maps each failure to the detector that can observe it. We run stub detectors over synthetic claim/reference pairs: an entity-registry check, a token-overlap proximity sensor, and an ordered-relation check. Each sensor catches one failure and is blind to another.


In [ ]:
registry = {"Company A", "Company B", "Company C", "Marie Curie"}


def entity_check(claim_entities):
    return "PASS" if all(e in registry for e in claim_entities) else "FLAG"


def overlap(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb)


def relation_check(ev_rel, cl_rel):
    return "PASS" if ev_rel == cl_rel else "FLAG"


pairs = [
    {"name": "invented_entity", "claim": "Dr. Elena Voss won the 2017 Northbridge Medal", "ref": "registry lists no Elena Voss", "entities": ["Dr. Elena Voss"], "ev_rel": ("X", "won", "Y"), "cl_rel": ("Elena Voss", "won", "Medal")},
    {"name": "relation_reversal", "claim": cl, "ref": ev, "entities": ["Company A", "Company B", "Company C"], "ev_rel": ("Company A", "acquired", "Company B"), "cl_rel": ("Company B", "acquired", "Company A")},
    {"name": "correct_paraphrase", "claim": "In 2024 Company A bought Company B from Company C", "ref": ev, "entities": ["Company A", "Company B", "Company C"], "ev_rel": ("Company A", "acquired", "Company B"), "cl_rel": ("Company A", "acquired", "Company B")},
]

print(f"{'pair':>18} | {'entity':>6} {'overlap':>7} {'relation':>8}")
for p in pairs:
    e = entity_check(p["entities"])
    o = overlap(p["claim"], p["ref"] if p["name"] == "invented_entity" else ev)
    r = relation_check(p["ev_rel"], p["cl_rel"])
    print(f"{p['name']:>18} | {e:>6} {o:7.3f} {r:>8}")

rev_overlap = overlap(ev, cl)
rev_entity = entity_check(["Company A", "Company B", "Company C"])
rev_relation = relation_check(("Company A", "acquired", "Company B"), ("Company B", "acquired", "Company A"))


In [ ]:
assert entity_check(["Dr. Elena Voss"]) == "FLAG"  # registry catches invented entity
assert rev_entity == "PASS"  # ... but entities are all real in the reversal: blind spot
assert rev_overlap > 0.80  # topical proximity stays high under role reversal
assert rev_relation == "FLAG"  # ordered-relation check separates what proximity cannot
print("blind spot confirmed: proximity + registry PASS the reversal; relation check FLAGs it")


## What we earned

- Two responses share the response-level label FAIL yet carry disjoint typed failures (invented_entity vs relation_reversal): the scalar hides the failure location.
- The reversal preserves every entity and keeps token overlap above 0.80, so the registry and proximity sensors pass it while the ordered-relation check flags it. The failure definition predicts the detector.

Next: Chapter 3 — Evidence, Truth, and Verifiability, which turns the typed record into an evidence-bearing Claim Verification Object.
